<a href="https://colab.research.google.com/github/mafloress/ProyectoFinalML1/blob/mlops-pipeline-setup/bank_marketing_project/feature_pipeline/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingeniería de Características para Datos de Marketing Bancario

## 1. Importar Bibliotecas Necesarias

In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
import requests # Añadido para descargas
import matplotlib.pyplot as plt # Para graficar
import seaborn as sns # Para visualizaciones mejoradas
from sklearn.preprocessing import StandardScaler # Para escalar características numéricas
from sklearn.feature_selection import mutual_info_classif, SelectKBest # Para Selección de Características
import json # Para guardar la lista de características seleccionadas

# Mostrar gráficos en línea
%matplotlib inline

# Establecer estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Descargar y Cargar los Datos

In [15]:
# prompt: 1.- Genera el código para descargar el archivo: 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip' 2.- obtener ubicación del archivo descargado.

url = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
zip_file_name = 'bank+marketing.zip'

# Descargar el archivo
response = requests.get(url)
if response.status_code == 200:
  with open(zip_file_name, 'wb') as f:
    f.write(response.content)
  print(f"Archivo descargado exitosamente como: {zip_file_name}")
else:
  print(f"Error al descargar el archivo. Código de estado: {response.status_code}")

# Obtener la ubicación del archivo descargado en el entorno de Colab
downloaded_file_path = os.path.join(os.getcwd(), zip_file_name)
print(f"Ubicación del archivo descargado: {downloaded_file_path}")


Archivo descargado exitosamente como: bank+marketing.zip
Ubicación del archivo descargado: /content/bank+marketing.zip


In [18]:
# prompt: Descomprime recursivamente el archivo de la variable downloaded_file_path y muestra resultados y obtiene la ubicacion del archivo bank-full.csv

def recursive_unzip(zip_path, extract_to='.'):
    """
    Recursively unzips files.

    Args:
        zip_path (str): Path to the zip file.
        extract_to (str): Directory to extract files to.
    """
    if not os.path.exists(extract_to):
        os.makedirs(extract_to)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
            print(f"Extracted: {zip_path} to {extract_to}")
            # Check if any extracted file is another zip file
            for member in zf.namelist():
                member_path = os.path.join(extract_to, member)
                if os.path.isfile(member_path) and member_path.lower().endswith('.zip'):
                    print(f"Found nested zip: {member_path}")
                    recursive_unzip(member_path, os.path.join(extract_to, os.path.splitext(member)[0]))
                    # Remove the nested zip after extraction
                    os.remove(member_path)
    except zipfile.BadZipFile:
        print(f"Error: {zip_path} is not a valid zip file.")
    except Exception as e:
        print(f"An error occurred while processing {zip_path}: {e}")

# Descomprimir el archivo descargado
extract_directory = 'extracted_data'
recursive_unzip(downloaded_file_path, extract_directory)

# Buscar la ubicación del archivo bank-full.csv
bank_full_csv_path = None
for root, dirs, files in os.walk(extract_directory):
    if 'bank-full.csv' in files:
        bank_full_csv_path = os.path.join(root, 'bank-full.csv')
        break

if bank_full_csv_path:
    print(f"\nUbicación del archivo bank-full.csv: {bank_full_csv_path}")
else:
    print("\nNo se encontró el archivo bank-full.csv.")


Extracted: /content/bank+marketing.zip to extracted_data
Found nested zip: extracted_data/bank.zip
Extracted: extracted_data/bank.zip to extracted_data/bank
Found nested zip: extracted_data/bank-additional.zip
Extracted: extracted_data/bank-additional.zip to extracted_data/bank-additional

Ubicación del archivo bank-full.csv: extracted_data/bank/bank-full.csv


In [26]:
# archivo del conjunto de datos
desired_csv_output_path = bank_full_csv_path


# Cargar bank-full.csv en un DataFrame de pandas
df_original = None # Mantener el df original como referencia si es necesario
if os.path.exists(desired_csv_output_path):
    try:
        df_original = pd.read_csv(desired_csv_output_path, sep=';')
        df = df_original.copy() # Trabajar con una copia para el preprocesamiento
        print(f"\n'{desired_csv_output_path}' cargado exitosamente en el DataFrame.")
    except Exception as e:
        print(f"Error cargando '{desired_csv_output_path}': {e}")
        df = None
else:
    print(f"Omitiendo carga del DataFrame ya que '{desired_csv_output_path}' no fue encontrado.")
    df = None


'extracted_data/bank/bank-full.csv' cargado exitosamente en el DataFrame.


## 3. Análisis Exploratorio de Datos Inicial (EDA)

In [27]:
if df is not None:
    print("Dimensiones del Conjunto de Datos:")
    print(df.shape)
else:
    print("DataFrame no cargado. Omitiendo EDA.")

Dimensiones del Conjunto de Datos:
(45211, 17)


In [28]:
if df is not None:
    print("\nPrimeras 5 filas:")
    display(df.head())


Primeras 5 filas:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [29]:
if df is not None:
    print("\nResumen conciso del DataFrame:")
    df.info()


Resumen conciso del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [6]:
if df is not None:
    print("\nEstadísticas descriptivas:")
    display(df.describe(include='all'))

In [7]:
if df is not None:
    print("\nConteo de valores faltantes (antes de manejar 'unknown'):")
    print(df.isnull().sum())

## 4. Identificar Características Numéricas y Categóricas

In [ ]:
if df is not None:
    numerical_features = df.select_dtypes(include=np.number).columns.tolist()
    categorical_features = df.select_dtypes(include='object').columns.tolist()

    if 'y' in categorical_features:
        target_column = 'y'
        categorical_features.remove(target_column)
    elif 'y' in numerical_features: # Si 'y' ya es numérica (ej. 0/1)
        target_column = 'y'
        # Aún la trataremos como objetivo, no como una característica numérica típica para escalar inicialmente
    else:
        target_column = None
        print("Advertencia: Columna objetivo 'y' no encontrada.")

    print("Características Numéricas Originales:", numerical_features)
    print("Características Categóricas Originales (excluyendo objetivo):", categorical_features)

## 5. EDA Detallado - Visualizaciones

### 5.1. Distribución de la Variable Objetivo

In [ ]:
if df is not None and target_column and target_column in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x=target_column, data=df)
    plt.title('Distribución de la Variable Objetivo (y)')
    plt.show()
    print(df[target_column].value_counts(normalize=True))

### 5.2. Análisis de Características Numéricas

In [ ]:
if df is not None and numerical_features:
    # Excluir el objetivo si era inicialmente numérico para estos gráficos generales
    plot_numerical_features = [f for f in numerical_features if f != target_column]
    if plot_numerical_features:
        print("\nHistogramas para Características Numéricas:")
        df[plot_numerical_features].hist(figsize=(12, 10), bins=20)
        plt.tight_layout()
        plt.show()

In [ ]:
if df is not None and numerical_features and target_column and target_column in df.columns:
    plot_numerical_features = [f for f in numerical_features if f != target_column]
    if plot_numerical_features:
        print("\nDiagramas de Caja (Boxplots) para Características Numéricas vs. Variable Objetivo 'y':")
        for feature in plot_numerical_features:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=target_column, y=feature, data=df)
            plt.title(f'{feature} vs. {target_column}')
            plt.show()

### 5.3. Análisis de Características Categóricas

In [ ]:
if df is not None and categorical_features:
    print("\nGráficos de Conteo para Características Categóricas:")
    for feature in categorical_features:
        if df[feature].nunique() < 20:
            plt.figure(figsize=(10, 6))
            sns.countplot(y=feature, data=df, order = df[feature].value_counts().index)
            plt.title(f'Distribución de {feature}')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Omitiendo gráfico para {feature} ya que tiene demasiados valores únicos ({df[feature].nunique()}).")

In [ ]:
if df is not None and categorical_features and target_column and target_column in df.columns:
    print("\nGráficos de Conteo para Características Categóricas vs. Variable Objetivo 'y':")
    for feature in categorical_features:
        if df[feature].nunique() < 20:
            plt.figure(figsize=(12, 7))
            sns.countplot(x=feature, hue=target_column, data=df)
            plt.title(f'{feature} vs. {target_column}')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Omitiendo gráfico para {feature} vs. {target_column} ya que {feature} tiene demasiados valores únicos ({df[feature].nunique()}).")

### 5.4. Análisis de Correlación

In [ ]:
if df is not None and numerical_features:
    # Convertir temporalmente el objetivo a numérico para correlación si aún no lo es
    df_corr = df.copy()
    if target_column and df_corr[target_column].dtype == 'object':
      df_corr[target_column] = df_corr[target_column].map({'yes': 1, 'no': 0}).fillna(-1) # codificación temporal para corr

    # Incluir el objetivo en las características numéricas para la matriz de correlación si está codificado binariamente
    corr_features = [f for f in numerical_features if f != target_column]
    if target_column and df_corr[target_column].dtype != 'object':
        corr_features.append(target_column)

    if corr_features:
        print("\nMatriz de Correlación (Características Numéricas y Objetivo Codificado):")
        corr_matrix = df_corr[corr_features].corr()
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
        plt.title('Matriz de Correlación')
        plt.show()
        display(corr_matrix)

## 6. Manejar valores 'unknown'

In [ ]:
if df is not None:
    print("Columnas con valores 'unknown' antes del manejo:")
    cols_with_unknown = []
    for col in df.columns:
        # Verificar si 'unknown' está en los valores únicos, manejando posibles tipos float
        if df[col].dtype == 'object' and 'unknown' in df[col].unique():
            cols_with_unknown.append(col)
            print(f"Columna '{col}': {df[col].value_counts()['unknown']} valores 'unknown'")

    if not cols_with_unknown:
        print("No se encontraron columnas con valores 'unknown'.")
    else:
        # Reemplazar 'unknown' con np.nan
        df.replace('unknown', np.nan, inplace=True)
        print("\nSe reemplazó 'unknown' con np.nan.")

    # Imputar valores NaN con la moda para cada columna
    print("\nImputando valores NaN con la moda...")
    for column in df.columns:
        if df[column].isnull().any():
            mode_val = df[column].mode()[0]
            df[column].fillna(mode_val, inplace=True)
            print(f"NaNs rellenados en '{column}' con la moda: {mode_val}")

    print("\nConteo de valores faltantes después de manejar 'unknown' e imputación:")
    print(df.isnull().sum().sum(), "valores faltantes totales.")

## 7. Detección y Manejo de Outliers

Los diagramas de caja para las características numéricas (por ejemplo, 'age', 'balance', 'duration', 'campaign', 'pdays', 'previous') indican la presencia de outliers.
- 'age': Algunos individuos mayores.
- 'balance': Número significativo de saldos positivos altos y algunos negativos (sobregiros).
- 'duration': Algunas duraciones de llamada muy largas. Nota: Esta característica está altamente correlacionada con el objetivo y debe manejarse con cuidado (por ejemplo, no usarla como entrada si es información posterior a la llamada, o si se usa, debe entenderse su naturaleza).
- 'campaign': Algunos clientes contactados muchas veces.
- 'pdays': Muchos valores -1 (no contactados previamente), y luego una dispersión para aquellos contactados.
- 'previous': Algunos clientes contactados múltiples veces antes.

**Estrategia para esta iteración:**
Para este pipeline inicial, notaremos estos outliers pero no implementaremos eliminación o transformación agresiva de outliers. Esto es para preservar la mayor cantidad de datos originales posible y establecer una línea base.

**Posibles estrategias futuras si los outliers resultan problemáticos:**
1.  **Capping/Winsorization**: Limitar valores extremos a un cierto percentil (por ejemplo, 1º y 99º).
2.  **Transformación**: Aplicar transformaciones logarítmicas, de raíz cuadrada o Box-Cox para reducir la asimetría y el impacto de los outliers.
3.  **Eliminación**: Si se considera que los outliers son errores o son extremadamente influyentes y no representativos, podrían eliminarse, pero esto debe hacerse con cautela.
4.  **Uso de modelos robustos**: Algunos modelos son inherentemente más robustos a los outliers.

## 8. Preprocesamiento de Datos - Codificación

### 8.1. Características Categóricas Binarias

In [ ]:
if df is not None and target_column:
    binary_cols = ['default', 'housing', 'loan'] # El objetivo 'y' se manejará por separado si no es ya 0/1
    if target_column in df.columns and df[target_column].dtype == 'object':
        binary_cols.append(target_column)

    for col in binary_cols:
        if col in df.columns and df[col].dtype == 'object': # Verificar si es tipo objeto antes de mapear
            # Asegurarse de que solo 'yes' y 'no' estén presentes, o manejar otros casos
            unique_vals = df[col].unique()
            if set(unique_vals) <= {'yes', 'no'}:
                 df[col] = df[col].map({'yes': 1, 'no': 0})
                 print(f"'{col}' codificado: {df[col].value_counts(dropna=False).to_dict()}")
            else:
                 print(f"Advertencia: La columna '{col}' no es estrictamente 'yes'/'no' y no fue codificada binariamente. Valores: {unique_vals}")
        elif col in df.columns and df[col].dtype != 'object':
             print(f"Columna '{col}' ya es numérica. Omitiendo codificación binaria.")
        else:
            print(f"Advertencia: Columna binaria '{col}' no encontrada en el DataFrame para codificación.")

    # Asegurar que el objetivo 'y' sea 0/1 y actualizar su estado si ocurrió la codificación
    if target_column and target_column in df.columns and df[target_column].dtype != 'object':
      if target_column not in numerical_features: numerical_features.append(target_column)
      if target_column in categorical_features: categorical_features.remove(target_column)

### 8.2. Otras Características Categóricas (One-Hot Encoding)

In [ ]:
if df is not None:
    # Re-identificar características categóricas después del manejo de 'unknown' y codificación binaria
    categorical_features_to_encode = df.select_dtypes(include='object').columns.tolist()
    # Asegurar que el objetivo no esté en esta lista si era objeto y ahora está codificado, o ya era numérico
    if target_column in categorical_features_to_encode:
        categorical_features_to_encode.remove(target_column)

    print(f"\nCaracterísticas categóricas para codificación one-hot: {categorical_features_to_encode}")

    if categorical_features_to_encode:
        df_encoded = pd.get_dummies(df, columns=categorical_features_to_encode, drop_first=True, dummy_na=False) # dummy_na=False es el valor por defecto
        print("\nDimensiones del DataFrame antes de one-hot encoding:", df.shape)
        print("Dimensiones del DataFrame después de one-hot encoding:", df_encoded.shape)
        # display(df_encoded.head())
        df = df_encoded # Actualizar df al nuevo dataframe codificado
    else:
        print("\nNo quedan características categóricas para codificación one-hot.")

    # Mostrar columnas finales para verificar el objetivo
    # print("\nColumnas finales del DataFrame:", df.columns.tolist())

## 9. Preprocesamiento de Datos - Escalado de Características Numéricas

In [ ]:
if df is not None and target_column and target_column in df.columns:
    current_numerical_features = df.select_dtypes(include=np.number).columns.tolist()

    features_to_scale = [col for col in current_numerical_features if col != target_column]

    print(f"\nCaracterísticas numéricas a escalar (excluyendo objetivo '{target_column}'): {features_to_scale}")

    if features_to_scale:
        scaler = StandardScaler()
        df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
        print("\nCaracterísticas numéricas escaladas usando StandardScaler.")
        # print("\nPrimeras 5 filas después del escalado:")
        # display(df.head())
    else:
        print("\nNo hay características numéricas para escalar.")
else:
    print("DataFrame o columna objetivo no disponible para escalado.")

## 10. Guardar Datos Procesados

In [ ]:
processed_file_path = None # Inicializar
if df is not None:
    processed_file_path = 'bank-full-processed.csv'
    df.to_csv(processed_file_path, index=False)
    print(f"\nDataFrame procesado guardado en '{processed_file_path}'")

    # Verificar cargándolo de nuevo (opcional)
    # df_loaded_processed = pd.read_csv(processed_file_path)
    # display(df_loaded_processed.head())
    # print(f"Dimensiones de los datos procesados cargados: {df_loaded_processed.shape}")
else:
    print("DataFrame no disponible. Omitiendo guardado de datos procesados.")

## 11. Selección de Características usando Ganancia de Información Mutua

Ahora que los datos están preprocesados (valores 'unknowns' manejados, codificados, escalados), aplicaremos selección de características para identificar las más relevantes para predecir la variable objetivo 'y'. Usaremos Ganancia de Información Mutua (Mutual Information Gain) para este propósito.

In [ ]:
df_for_selection = None
if processed_file_path and os.path.exists(processed_file_path):
    try:
        df_for_selection = pd.read_csv(processed_file_path)
        print(f"'{processed_file_path}' cargado para selección de características. Dimensiones: {df_for_selection.shape}")
    except Exception as e:
        print(f"Error cargando '{processed_file_path}': {e}")
        df_for_selection = None
elif df is not None: # Recurrir a usar el df en memoria si falló el guardado/carga de archivo
    print("Usando DataFrame de la memoria para selección de características.")
    df_for_selection = df.copy()
else:
    print("Datos preprocesados no disponibles para selección de características.")

X = None
y_fs = None # y para selección de características

if df_for_selection is not None and target_column and target_column in df_for_selection.columns:
    # Asegurar que el tipo de 'y' sea entero para mutual_info_classif
    if df_for_selection[target_column].dtype == 'float': # Puede suceder si era binario (0.0/1.0)
        df_for_selection[target_column] = df_for_selection[target_column].astype(int)
        print(f"Columna objetivo '{target_column}' convertida a entero para selección de características.")

    if df_for_selection[target_column].isnull().any():
        print(f"Advertencia: La columna objetivo '{target_column}' contiene valores NaN. Esto podría afectar la selección de características.")
        # Opcionalmente, manejar NaNs aquí, ej. eliminando filas con NaN en el objetivo
        # df_for_selection.dropna(subset=[target_column], inplace=True)
        # print(f"Filas con NaN en el objetivo eliminadas. Nuevas dimensiones: {df_for_selection.shape}")

    X = df_for_selection.drop(columns=[target_column])
    y_fs = df_for_selection[target_column]
    print(f"X preparado (dimensiones de características: {X.shape}) e y (dimensiones de objetivo: {y_fs.shape}) para selección de características.")

    # Asegurar que todas las características en X sean numéricas (deberían serlo después del preprocesamiento)
    non_numeric_cols = X.select_dtypes(exclude=np.number).columns.tolist()
    if non_numeric_cols:
        print(f"Advertencia: Columnas no numéricas encontradas en X: {non_numeric_cols}. Esto causará un error en mutual_info_classif.")
        # Intentar convertir, o eliminar, o lanzar error
        # Por ahora, asumamos que deberían haber sido manejadas y proceder; ocurrirá un error si no es así.
else:
    print("La selección de características no puede proceder ya que los datos o la columna objetivo no están configurados correctamente.")

In [ ]:
mi_scores = None
if X is not None and y_fs is not None:
    try:
        # Calcular puntuaciones de información mutua
        # Asegurar que y_fs no contenga NaNs si no se manejaron antes
        if y_fs.isnull().any():
            print("La variable objetivo y_fs contiene NaNs. Limpiando antes de mutual_info_classif...")
            X_temp = X[~y_fs.isnull()]
            y_fs_temp = y_fs[~y_fs.isnull()]
            if y_fs_temp.empty:
                raise ValueError("La variable objetivo y_fs está toda en NaNs después de la limpieza.")
            print(f"Dimensiones después de eliminar NaN del objetivo para cálculo de MI: X_temp={X_temp.shape}, y_fs_temp={y_fs_temp.shape}")
            mi_scores = mutual_info_classif(X_temp, y_fs_temp, random_state=42)
        else:
            mi_scores = mutual_info_classif(X, y_fs, random_state=42)

        mi_scores_series = pd.Series(mi_scores, name='MI_Score', index=X.columns)
        mi_scores_series = mi_scores_series.sort_values(ascending=False)

        print("\nPuntuaciones de Información Mutua (Top 30):")
        display(mi_scores_series.head(30))

        # Graficando las puntuaciones de MI
        plt.figure(figsize=(12, 8))
        mi_scores_series.head(30).plot(kind='bar')
        plt.title('Top 30 Características por Puntuación de Información Mutua')
        plt.ylabel('Puntuación de Información Mutua')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error durante el cálculo o graficación de la información mutua: {e}")
        # print traceback para más detalles si se ejecuta interactivamente
        # import traceback
        # traceback.print_exc()
else:
    print("X o y_fs no disponibles para el cálculo de información mutua.")

In [ ]:
selected_features_df = None
selected_features_list = []
K = 20 # Número deseado de características principales

if mi_scores_series is not None and X is not None and y_fs is not None:
    num_total_features = X.shape[1]
    k_to_select = min(K, num_total_features) # Seleccionar K o todas si hay menos de K

    print(f"\nSeleccionando las {k_to_select} características principales basadas en puntuaciones MI...")

    # Usando SelectKBest con mutual_info_classif
    # Necesidad de manejar NaNs potenciales en y_fs también para SelectKBest
    if y_fs.isnull().any():
        print("La variable objetivo y_fs contiene NaNs. Limpiando antes de SelectKBest...")
        X_select = X[~y_fs.isnull()].copy() # Usar .copy() para evitar SettingWithCopyWarning
        y_select = y_fs[~y_fs.isnull()].copy()
        if y_select.empty:
            print("Error: La variable objetivo y_fs está toda en NaNs después de la limpieza. No se puede proceder con SelectKBest.")
            selector = None
        else:
            print(f"Dimensiones después de eliminar NaN del objetivo para SelectKBest: X_select={X_select.shape}, y_select={y_select.shape}")
            selector = SelectKBest(score_func=mutual_info_classif, k=k_to_select)
            selector.fit(X_select, y_select)
            selected_features_mask = selector.get_support()
            selected_features_list = X_select.columns[selected_features_mask].tolist()
    else:
        selector = SelectKBest(score_func=mutual_info_classif, k=k_to_select)
        selector.fit(X, y_fs)
        selected_features_mask = selector.get_support()
        selected_features_list = X.columns[selected_features_mask].tolist()

    if selected_features_list:
        print(f"\n{len(selected_features_list)} características seleccionadas:")
        print(selected_features_list)

        # Crear un nuevo DataFrame con características seleccionadas y el objetivo
        # Usar X e y_fs originales para asegurar que todas las filas se incluyan, luego eliminar NaNs si y_fs los tenía inicialmente
        selected_features_df = X[selected_features_list].copy() # .copy() para evitar SettingWithCopyWarning
        selected_features_df[target_column] = y_fs.values # Añadir variable objetivo de vuelta

        # Si y_fs tenía NaNs, esas filas tendrán NaN en el objetivo en selected_features_df
        # Dependiendo de la estrategia, se podrían eliminar ahora o dejar que el pipeline de entrenamiento los maneje
        # Por ahora, mantenerlos para coincidir con el conteo original de filas; el pipeline de entrenamiento debe ser consciente
        # if y_fs.isnull().any():
        #    print(f"La columna objetivo tenía NaNs. El selected_features_df resultante podría tener NaNs en el objetivo. Dimensiones: {selected_features_df.shape}")

        print("\nPrimeras 5 filas del DataFrame con características seleccionadas y objetivo:")
        display(selected_features_df.head())
    else:
        print("No se seleccionaron características. Esto podría indicar un problema.")
else:
    print("Puntuaciones MI o datos no disponibles. Omitiendo selección de características basada en KBest.")

## 12. Guardar Datos de Características Seleccionadas Finales y Lista

In [ ]:
selected_features_file_path = None
selected_features_list_path = None

if selected_features_df is not None:
    selected_features_file_path = 'bank-features-selected.csv'
    selected_features_df.to_csv(selected_features_file_path, index=False)
    print(f"\nDataFrame con características seleccionadas guardado en '{selected_features_file_path}'")
    print(f"Dimensiones de los datos de características seleccionadas guardados: {selected_features_df.shape}")

    # Guardar también la lista de nombres de características seleccionadas (ej. en un archivo JSON)
    if selected_features_list:
        selected_features_list_path = 'selected_feature_names.json'
        with open(selected_features_list_path, 'w') as f:
            json.dump(selected_features_list, f)
        print(f"Lista de nombres de características seleccionadas guardada en '{selected_features_list_path}'")
else:
    print("DataFrame con características seleccionadas no disponible. Omitiendo guardado.")

### Resumen de Selección de Características:
Empleamos Ganancia de Información Mutua para evaluar la relevancia de cada característica con respecto a la variable objetivo 'y'. La información mutua mide la cantidad de información obtenida sobre una variable aleatoria al observar la otra. Valores más altos indican una relación más fuerte.

Basándonos en estas puntuaciones, seleccionamos las K características principales. En esta iteración, K se estableció en 20 (o menos si el número total de características era menor a 20). El DataFrame resultante, que contiene solo estas características seleccionadas y la variable objetivo, se ha guardado en `bank-features-selected.csv`. También se ha guardado un archivo JSON `selected_feature_names.json` que contiene la lista de estos nombres de características. Este conjunto de datos está ahora preparado para la fase de entrenamiento del modelo.

## 13. Observaciones y Resumen del Preprocesamiento y Selección de Características

### Hallazgos del EDA:
- **Variable Objetivo ('y')**: El conjunto de datos está desequilibrado (aprox. 88% 'no', 12% 'yes').
- **Características Numéricas**: Distribuciones variadas, presencia de outliers (ej. 'balance', 'duration'). 'duration' está altamente correlacionada con el objetivo pero podría ser una variable de fuga (leakage variable).
- **Características Categóricas**: Se encontraron y manejaron valores 'unknown'. Las distribuciones variaron. Se observaron relaciones con 'y'.

### Pasos de Preprocesamiento Realizados:
1.  **Datos Cargados**: `bank-full.csv`.
2.  **Manejo de Valores 'unknown'**: Reemplazados con `np.nan`, luego imputados con la moda.
3.  **Características Binarias Codificadas**: 'default', 'housing', 'loan', 'y' a 1/0.
4.  **Características Categóricas Codificadas con One-Hot**: Usando `pd.get_dummies(drop_first=True)`.
5.  **Características Numéricas Escaladas**: Usando `StandardScaler` (excluyendo el objetivo 'y').
6.  **Datos Procesados Guardados**: En `bank-full-processed.csv`.

### Pasos de Selección de Características:
1.  **Cálculo de Puntuaciones de Información Mutua**: Entre cada característica y el objetivo 'y' usando los datos preprocesados.
2.  **Selección de las K Características Principales**: Se apuntó a K=20 características con las puntuaciones MI más altas. Si el total de características < 20, se seleccionaron todas. El número real de características seleccionadas se reporta en la salida anterior.
3.  **Datos de Características Seleccionadas Guardados**: El DataFrame con características seleccionadas y el objetivo guardado en `bank-features-selected.csv`.
4.  **Nombres de Características Seleccionadas Guardados**: La lista de nombres de características seleccionadas guardada en `selected_feature_names.json`.

### Próximos Pasos:
El conjunto de datos `bank-features-selected.csv` que contiene las características más relevantes está ahora listo para el notebook `model_training.ipynb`.